## Phần 1 – Cài đặt & Fine-tune SAM-Med2D

In [ ]:
%cd /content
!git clone https://github.com/OpenGVLab/SAM-Med2D/
%cd /content/SAM-Med2D
!pip install -e . -q

In [ ]:
# Tạo folder checkpoint trong SAM-Med2D
!mkdir -p /content/SAM-Med2D/checkpoint

# Tải trọng số pretrained từ Google Drive
!gdown "https://drive.google.com/uc?id=1ARiB5RkSsWmAB_8mqWnwDF8ZKTtFwsjl" \
    -O /content/SAM-Med2D/checkpoint/sam_med2d.pth

!ls -lh /content/SAM-Med2D/checkpoint

In [ ]:
# =========================================================
# DOWNLOAD DATASET ZIP TỪ GOOGLE DRIVE
# =========================================================

!gdown --id 1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW  # dataset_BTXRD (1 ảnh 1 mask) — dùng chung với PGA

!unzip -oq dataset_BTXRD.zip

# Xóa thư mục cũ nếu có
!rm -rf dataset_BTXRD
# Sau đó giải nén
!unzip -oq dataset_BTXRD.zip

In [ ]:
# ── Chuyển đổi: JSON annotation → per-polygon mask PNG ──────────────────────
# Mỗi polygon (khối u) → 1 file mask riêng (IMG001768_1.png, IMG001768_2.png...)
# Sau bước này: train=1859 samples, val=211, test=248 (khớp chính xác với PGA)

import cv2, json, numpy as np, os, glob

%cd /content/SAM-Med2D

def build_per_polygon_masks(split):
    ann_dir  = f"dataset_BTXRD/{split}/annotations"
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"

    if not os.path.exists(ann_dir):
        print(f"[!] {split}: không có thư mục annotations, bỏ qua."); return

    # Xóa toàn bộ mask cũ (merged 1-per-image) để tránh đếm nhầm
    old = glob.glob(f"{mask_dir}/*.png")
    for p in old: os.remove(p)
    print(f"[*] {split}: đã xóa {len(old)} mask cũ")

    n_written = 0
    for json_path in sorted(glob.glob(f"{ann_dir}/*.json")):
        base = os.path.splitext(os.path.basename(json_path))[0]

        # Tìm file ảnh gốc (.png / .jpg)
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg']:
            p = os.path.join(img_dir, base + ext)
            if os.path.exists(p): img_path = p; break
        if img_path is None:
            print(f"  [!] Không tìm thấy ảnh cho {base}"); continue

        img  = cv2.imread(img_path)
        H, W = img.shape[:2]
        data = json.load(open(json_path))

        polygons = [s for s in data.get('shapes', [])
                    if s.get('shape_type') == 'polygon']

        for i, shape in enumerate(polygons, start=1):
            pts  = np.array(shape['points'], dtype=np.float32)
            pts  = pts.reshape((-1, 1, 2)).astype(np.int32)
            mask = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            out  = os.path.join(mask_dir, f"{base}_{i}.png")
            cv2.imwrite(out, mask)
            n_written += 1

    print(f"[✅] {split}: tạo {n_written} per-polygon mask files")

for split in ['train', 'val', 'test']:
    build_per_polygon_masks(split)

print("\n✅ Xong! Số samples mới:")
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f"dataset_BTXRD/{split}/masks/*.png"))
    print(f"  {split}: {n} mask files")


In [ ]:
%cd /content/SAM-Med2D
import os, json

def create_mapping_json(split):
    img_dir = f"dataset_BTXRD/{split}/images"
    mask_dir = f"dataset_BTXRD/{split}/masks"
    image2label = {}

    if not os.path.exists(img_dir) or not os.path.exists(mask_dir):
        print(f"[!] Không tìm thấy thư mục cho tập {split}")
        return

    all_masks = os.listdir(mask_dir) if os.path.exists(mask_dir) else []

    for img_name in os.listdir(img_dir):
        if not img_name.endswith(('.png', '.jpg', '.jpeg')): continue
        base = os.path.splitext(img_name)[0]
        img_path = os.path.abspath(os.path.join(img_dir, img_name))
        matched_masks = [os.path.abspath(os.path.join(mask_dir, m))
                         for m in all_masks
                         if os.path.splitext(m)[0] == base
                         or os.path.splitext(m)[0].startswith(f"{base}_")]
        if matched_masks:
            image2label[img_path] = sorted(matched_masks)

    json_out = f"dataset_BTXRD/image2label_{split}.json"
    with open(json_out, 'w', encoding='utf-8') as f:
        json.dump(image2label, f, indent=4)
    print(f"[*] Đã tạo map cho {split}: {len(image2label)} ảnh → {json_out}")

create_mapping_json("train")
create_mapping_json("val")

# Tạo label2image_val.json (cần cho val_one_epoch trong train.py)
def create_evaluation_json(split="test"):
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"
    label2image = {}

    if not os.path.exists(mask_dir) or not os.path.exists(img_dir):
        print(f"[!] Không tìm thấy thư mục cho tập: '{split}'"); return

    img_dict = {}
    for f in os.listdir(img_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_dict[os.path.splitext(f)[0]] = f

    for mask_name in os.listdir(mask_dir):
        if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        mask_base = os.path.splitext(mask_name)[0]
        img_base  = mask_base.split("_")[0] if "_" in mask_base else mask_base
        if img_base in img_dict:
            label2image[f"dataset_BTXRD/{split}/masks/{mask_name}"] =                         f"dataset_BTXRD/{split}/images/{img_dict[img_base]}"

    out = f"dataset_BTXRD/label2image_{split}.json"
    with open(out, "w", encoding="utf-8") as f_out:
        json.dump(label2image, f_out, indent=4, ensure_ascii=False)
    print(f"[*] label2image_{split}.json — {len(label2image)} mẫu → {out}")

create_evaluation_json("test")
create_evaluation_json("val")


In [ ]:
%%writefile /content/SAM-Med2D/train.py
# train.py — SAM-Med2D original (theo tác giả)
# Fix 1: apex import optional (không cần cài Apex trên Colab)
# Fix 2: randint crash khi iter_point < 3

from segment_anything import sam_model_registry, SamPredictor
import torch.nn as nn
import torch
import argparse
import os
from torch import optim
from torch.utils.data import DataLoader
from DataLoader import TrainingDataset, TestingDataset, stack_dict_batched
from torch.nn import functional as F
from utils import FocalDiceloss_IoULoss, get_logger, generate_point, setting_prompt_none
from metrics import SegMetrics
import time
from tqdm import tqdm
import numpy as np
import datetime# FIX 1: apex optional — chỉ cần khi dùng --use_amp
try:
    from apex import amp
    APEX_AVAILABLE = True
except ImportError:
    APEX_AVAILABLE = False
import random


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir", type=str, default="workdir", help="work dir")
    parser.add_argument("--run_name", type=str, default="sam-med2d", help="run model name")
    parser.add_argument("--epochs", type=int, default=50, help="number of epochs")
    parser.add_argument("--early_stop", type=int, default=10, help="early stopping patience")
    parser.add_argument("--batch_size", type=int, default=2, help="train batch size")
    parser.add_argument("--image_size", type=int, default=256, help="image_size")
    parser.add_argument("--mask_num", type=int, default=5, help="get mask number")
    parser.add_argument("--data_path", type=str, default="data_demo", help="train data path")
    parser.add_argument("--val_data_path", type=str, default=None, help="val data path (default same as data_path)")
    parser.add_argument("--zoom_ratio", type=float, nargs=2, default=[0.15, 0.45],
                        help="noisy bbox zoom range – cùng setup PGA")
    parser.add_argument("--shift_ratio", type=float, default=0.30,
                        help="noisy bbox shift ratio")
    parser.add_argument("--metrics", nargs='+', default=['iou', 'dice'], help="metrics")
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument("--lr", type=float, default=1e-4, help="learning rate")
    parser.add_argument("--resume", type=str, default=None, help="load resume")
    parser.add_argument("--model_type", type=str, default="vit_b", help="sam model_type")
    parser.add_argument("--sam_checkpoint", type=str, default="pretrain_model/sam-med2d_b.pth", help="sam checkpoint")
    parser.add_argument("--iter_point", type=int, default=8, help="point iterations")
    parser.add_argument('--lr_scheduler', type=str, default=None, help='lr scheduler')
    parser.add_argument("--point_list", type=list, default=[1, 3, 5, 9], help="point_list")
    parser.add_argument("--multimask", type=bool, default=True, help="ouput multimask")
    parser.add_argument("--encoder_adapter", type=bool, default=True, help="use adapter")
    parser.add_argument("--use_amp", type=bool, default=False, help="use amp")
    args = parser.parse_args()
    if args.resume is not None:
        args.sam_checkpoint = None
    return args


def to_device(batch_input, device):
    device_input = {}
    for key, value in batch_input.items():
        if value is not None:
            if key=='image' or key=='label':
                device_input[key] = value.float().to(device)
            elif type(value) is list or type(value) is torch.Size:
                 device_input[key] = value
            else:
                device_input[key] = value.to(device)
        else:
            device_input[key] = value
    return device_input


def prompt_and_decoder(args, batched_input, model, image_embeddings, decoder_iter=False):
    if batched_input["point_coords"] is not None:
        points = (batched_input["point_coords"], batched_input["point_labels"])
    else:
        points = None

    if decoder_iter:
        with torch.no_grad():
            sparse_embeddings, dense_embeddings = model.prompt_encoder(
                points=points,
                boxes=batched_input.get("boxes", None),
                masks=batched_input.get("mask_inputs", None),
            )
    else:
        sparse_embeddings, dense_embeddings = model.prompt_encoder(
            points=points,
            boxes=batched_input.get("boxes", None),
            masks=batched_input.get("mask_inputs", None),
        )

    low_res_masks, iou_predictions = model.mask_decoder(
        image_embeddings=image_embeddings,
        image_pe=model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=args.multimask,
    )

    if args.multimask:
        max_values, max_indexs = torch.max(iou_predictions, dim=1)
        max_values = max_values.unsqueeze(1)
        iou_predictions = max_values
        low_res = []
        for i, idx in enumerate(max_indexs):
            low_res.append(low_res_masks[i:i+1, idx])
        low_res_masks = torch.stack(low_res, 0)

    masks = F.interpolate(low_res_masks, (args.image_size, args.image_size),
                          mode="bilinear", align_corners=False)
    return masks, low_res_masks, iou_predictions


def train_one_epoch(args, model, optimizer, train_loader, epoch, criterion):
    train_loader = tqdm(train_loader)
    train_losses = []
    train_iter_metrics = [0] * len(args.metrics)

    for batch, batched_input in enumerate(train_loader):
        batched_input = stack_dict_batched(batched_input)
        batched_input = to_device(batched_input, args.device)

        # Tác giả gốc: ngẫu nhiên bbox HOẶC point làm prompt đầu vào
        if random.random() > 0.5:
            batched_input["point_coords"] = None
            flag = "boxes"
        else:
            batched_input["boxes"] = None
            flag = "point"

        # Freeze encoder backbone, chỉ train Adapter
        for n, value in model.image_encoder.named_parameters():
            if "Adapter" in n:
                value.requires_grad = True
            else:
                value.requires_grad = False

        labels = batched_input["label"]
        image_embeddings = model.image_encoder(batched_input["image"])

        B, _, _, _ = image_embeddings.shape
        image_embeddings_repeat = []
        for i in range(B):
            image_embed = image_embeddings[i].repeat(args.mask_num, 1, 1, 1)
            image_embeddings_repeat.append(image_embed)
        image_embeddings = torch.cat(image_embeddings_repeat, dim=0)

        masks, low_res_masks, iou_predictions = prompt_and_decoder(
            args, batched_input, model, image_embeddings, decoder_iter=False)
        loss = criterion(masks, labels, iou_predictions)
        loss.backward(retain_graph=False)
        optimizer.step()
        optimizer.zero_grad()

        if int(batch + 1) % 50 == 0:
            print(f'Epoch: {epoch+1}, Batch: {batch+1}, first {flag} prompt: {SegMetrics(masks, labels, args.metrics)}')

        # --- Vòng lặp refinement (tác giả gốc: iter_point lần) ---
        point_num = random.choice(args.point_list)
        batched_input = generate_point(masks, labels, low_res_masks, batched_input, point_num)
        batched_input = to_device(batched_input, args.device)

        image_embeddings = image_embeddings.detach().clone()
        for n, value in model.named_parameters():
            if "image_encoder" in n:
                value.requires_grad = False
            else:
                value.requires_grad = True

        # FIX 2: randint crash khi iter_point < 3 — bảo vệ lower bound
        init_mask_num = np.random.randint(1, max(2, args.iter_point - 1))
        for iter in range(args.iter_point):
            if iter == init_mask_num or iter == args.iter_point - 1:
                batched_input = setting_prompt_none(batched_input)

            masks, low_res_masks, iou_predictions = prompt_and_decoder(
                args, batched_input, model, image_embeddings, decoder_iter=True)
            loss = criterion(masks, labels, iou_predictions)
            loss.backward(retain_graph=True)
            optimizer.step()
            optimizer.zero_grad()

            if iter != args.iter_point - 1:
                point_num = random.choice(args.point_list)
                batched_input = generate_point(masks, labels, low_res_masks, batched_input, point_num)
                batched_input = to_device(batched_input, args.device)

            if int(batch + 1) % 50 == 0:
                if iter == init_mask_num or iter == args.iter_point - 1:
                    print(f'Epoch: {epoch+1}, Batch: {batch+1}, mask prompt: {SegMetrics(masks, labels, args.metrics)}')
                else:
                    print(f'Epoch: {epoch+1}, Batch: {batch+1}, point {point_num} prompt: {SegMetrics(masks, labels, args.metrics)}')

        if int(batch + 1) % 200 == 0:
            print(f"epoch:{epoch+1}, iteration:{batch+1}, loss:{loss.item()}")

        train_losses.append(loss.item())
        train_loader.set_postfix(train_loss=loss.item())

        train_batch_metrics = SegMetrics(masks, labels, args.metrics)
        train_iter_metrics = [train_iter_metrics[i] + train_batch_metrics[i]
                               for i in range(len(args.metrics))]

    return train_losses, train_iter_metrics



def val_one_epoch(args, model, val_loader):
    """Single-pass bbox inference trên val set — giống test.py, so sánh tại 256×256."""
    model.eval()
    val_iter_metrics = [0] * len(args.metrics)
    l = len(val_loader)
    for batched_input in val_loader:
        batched_input = to_device(batched_input, args.device)
        with torch.no_grad():
            image_embeddings = model.image_encoder(batched_input["image"])
            batched_input["point_coords"] = None
            batched_input["point_labels"] = None
            masks, _, iou_predictions = prompt_and_decoder(
                args, batched_input, model, image_embeddings, decoder_iter=False)
        # So sánh tại image_size (256) — nhất quán với resolution model xử lý
        ori = F.interpolate(batched_input["ori_label"].float(),
                            (args.image_size, args.image_size), mode='nearest')
        bm = SegMetrics(masks, ori, args.metrics)
        for j in range(len(args.metrics)):
            val_iter_metrics[j] += float(bm[j])
    return [m / l for m in val_iter_metrics]

def main(args):
    model = sam_model_registry[args.model_type](args).to(args.device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    criterion = FocalDiceloss_IoULoss()

    if args.lr_scheduler:
        scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)
        print('*******Use MultiStepLR')

    if args.resume is not None:
        with open(args.resume, "rb") as f:
            checkpoint = torch.load(f, weights_only=False)
            model.load_state_dict(checkpoint['model'])
            optimizer.load_state_dict(checkpoint['optimizer'].state_dict())
            print(f"*******load {args.resume}")

    print('*******Do not use mixed precision')

    train_dataset = TrainingDataset(args.data_path, image_size=args.image_size,
                                    mode='train', point_num=1, mask_num=args.mask_num,
                                    requires_name=False,
                                    zoom_ratio=tuple(args.zoom_ratio),
                                    shift_ratio=args.shift_ratio)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                               shuffle=True, num_workers=2)
    print('*******Train data:', len(train_dataset))

    # Validation set (single-pass bbox, mode zoom_out)
    val_data = args.val_data_path or args.data_path
    val_dataset = TestingDataset(val_data, image_size=args.image_size,
                                 mode='val', requires_name=False,
                                 return_ori_mask=True, prompt_mode='zoom_out')
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)
    print('*******Val data:', len(val_dataset))

    loggers = get_logger(os.path.join(args.work_dir, "logs",
                         f"{args.run_name}_{datetime.datetime.now().strftime('%Y%m%d-%H%M.log')}"))

    best_loss = 1e10
    best_val_dice = -1.0
    no_improve = 0
    early_stop = getattr(args, 'early_stop', 10)
    l = len(train_loader)

    for epoch in range(0, args.epochs):
        model.train()
        start = time.time()
        os.makedirs(os.path.join(f"{args.work_dir}/models", args.run_name), exist_ok=True)
        train_losses, train_iter_metrics = train_one_epoch(
            args, model, optimizer, train_loader, epoch, criterion)

        if args.lr_scheduler is not None:
            scheduler.step()

        train_iter_metrics = [metric / l for metric in train_iter_metrics]
        train_metrics = {args.metrics[i]: '{:.4f}'.format(train_iter_metrics[i])
                         for i in range(len(train_iter_metrics))}

        average_loss = np.mean(train_losses)
        lr = scheduler.get_last_lr()[0] if args.lr_scheduler is not None else args.lr
        loggers.info(f"epoch: {epoch + 1}, lr: {lr}, Train loss: {average_loss:.4f}, metrics: {train_metrics}")

        # Validation: single-pass bbox (giống test.py) — dùng để lưu best checkpoint
        val_metrics = val_one_epoch(args, model, val_loader)
        val_dice = val_metrics[args.metrics.index('dice')] if 'dice' in args.metrics else val_metrics[0]
        val_metrics_str = {args.metrics[i]: f"{val_metrics[i]:.4f}" for i in range(len(args.metrics))}
        loggers.info(f"  → Val (single-pass bbox): {val_metrics_str}")
        print(f"  → Val Dice={val_dice:.4f} | Best={best_val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_loss = average_loss  # track for reference
            no_improve = 0
            save_path = os.path.join(args.work_dir, "models", args.run_name, "best_sam.pth")
            state = {'model': model.float().state_dict(), 'optimizer': optimizer}
            torch.save(state, save_path)
            print(f"[Best] epoch {epoch+1}, val_dice={val_dice:.4f} → saved best_sam.pth")
        else:
            no_improve += 1

        end = time.time()
        print("Run epoch time: %.2fs" % (end - start))

        if no_improve >= early_stop:
            loggers.info(f"Early stopping at epoch {epoch+1} (no improve for {early_stop} epochs)")
            break

    # Lưu trọng số epoch cuối cùng (last)
    save_path = os.path.join(args.work_dir, "models", args.run_name, "last_sam.pth")
    state = {'model': model.float().state_dict(), 'optimizer': optimizer}
    torch.save(state, save_path)
    print(f"[Last] saved last_sam.pth")


if __name__ == '__main__':
    args = parse_args()
    main(args)

In [ ]:
%%writefile /content/SAM-Med2D/segment_anything/build_sam.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.

# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

import torch
from functools import partial
from .modeling import ImageEncoderViT, MaskDecoder, PromptEncoder, Sam, TwoWayTransformer
from torch.nn import functional as F

def build_sam_vit_h(args):
    return _build_sam(
        encoder_embed_dim=1280,
        encoder_depth=32,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[7, 15, 23, 31],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,
    )


build_sam = build_sam_vit_h


def build_sam_vit_l(args):
    return _build_sam(
        encoder_embed_dim=1024,
        encoder_depth=24,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[5, 11, 17, 23],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,
    )


def build_sam_vit_b(args):
    return _build_sam(
        encoder_embed_dim=768,
        encoder_depth=12,
        encoder_num_heads=12,
        encoder_global_attn_indexes=[2, 5, 8, 11],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,

    )


sam_model_registry = {
    "default": build_sam_vit_h,
    "vit_h": build_sam_vit_h,
    "vit_l": build_sam_vit_l,
    "vit_b": build_sam_vit_b,
}


def _build_sam(
    encoder_embed_dim,
    encoder_depth,
    encoder_num_heads,
    encoder_global_attn_indexes,
    image_size,
    checkpoint,
    encoder_adapter,
):
    prompt_embed_dim = 256
    image_size = image_size
    vit_patch_size = 16
    image_embedding_size = image_size // vit_patch_size
    sam = Sam(
        image_encoder=ImageEncoderViT(
            depth=encoder_depth,
            embed_dim=encoder_embed_dim,
            img_size=image_size,
            mlp_ratio=4,
            norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
            num_heads=encoder_num_heads,
            patch_size=vit_patch_size,
            qkv_bias=True,
            use_rel_pos = True,
            global_attn_indexes=encoder_global_attn_indexes,
            window_size=14,
            out_chans=prompt_embed_dim,
            adapter_train = encoder_adapter,
        ),
        prompt_encoder=PromptEncoder(
            embed_dim=prompt_embed_dim,
            image_embedding_size=(image_embedding_size, image_embedding_size),
            input_image_size=(image_size, image_size),
            mask_in_chans=16,
        ),
        mask_decoder=MaskDecoder(
            num_multimask_outputs=3,
            transformer=TwoWayTransformer(
                depth=2,
                embedding_dim=prompt_embed_dim,
                mlp_dim=2048,
                num_heads=8,
            ),
            transformer_dim=prompt_embed_dim,
            iou_head_depth=3,
            iou_head_hidden_dim=256,
        ),
        pixel_mean=[123.675, 116.28, 103.53],
        pixel_std=[58.395, 57.12, 57.375],
    )
    # sam.train()
    if checkpoint is not None:
        with open(checkpoint, "rb") as f:
            # Ép gán weights_only=False để PyTorch mới chịu đọc file chứa Adam optimizer của tác giả
            state_dict = torch.load(f, map_location="cpu", weights_only=False)
        try:
            if 'model' in state_dict.keys():
                print(encoder_adapter)
                sam.load_state_dict(state_dict['model'], False) # Tác giả đã để strict=False ở đây sẵn rồi
            else:
                if image_size==1024 and encoder_adapter==True:
                    sam.load_state_dict(state_dict, False)
                else:
                    sam.load_state_dict(state_dict)
        except:
            print('*******interpolate')
            new_state_dict = load_from(sam, state_dict, image_size, vit_patch_size)
            sam.load_state_dict(new_state_dict)
        print(f"*******load {checkpoint}")

    return sam


def load_from(sam, state_dicts, image_size, vit_patch_size):

    sam_dict = sam.state_dict()
    except_keys = ['mask_tokens', 'output_hypernetworks_mlps', 'iou_prediction_head']
    new_state_dict = {k: v for k, v in state_dicts.items() if
                      k in sam_dict.keys() and except_keys[0] not in k and except_keys[1] not in k and except_keys[2] not in k}
    pos_embed = new_state_dict['image_encoder.pos_embed']
    token_size = int(image_size // vit_patch_size)
    if pos_embed.shape[1] != token_size:
        # resize pos embedding, which may sacrifice the performance, but I have no better idea
        pos_embed = pos_embed.permute(0, 3, 1, 2)  # [b, c, h, w]
        pos_embed = F.interpolate(pos_embed, (token_size, token_size), mode='bilinear', align_corners=False)
        pos_embed = pos_embed.permute(0, 2, 3, 1)  # [b, h, w, c]
        new_state_dict['image_encoder.pos_embed'] = pos_embed
        rel_pos_keys = [k for k in sam_dict.keys() if 'rel_pos' in k]

        global_rel_pos_keys = [k for k in rel_pos_keys if
                                                        '2' in k or
                                                        '5' in k or
                                                        '7' in k or
                                                        '8' in k or
                                                        '11' in k or
                                                        '13' in k or
                                                        '15' in k or
                                                        '23' in k or
                                                        '31' in k]
        # print(sam_dict)
        for k in global_rel_pos_keys:
            h_check, w_check = sam_dict[k].shape
            rel_pos_params = new_state_dict[k]
            h, w = rel_pos_params.shape
            rel_pos_params = rel_pos_params.unsqueeze(0).unsqueeze(0)
            if h != h_check or w != w_check:
                rel_pos_params = F.interpolate(rel_pos_params, (h_check, w_check), mode='bilinear', align_corners=False)

            new_state_dict[k] = rel_pos_params[0, 0, ...]

    sam_dict.update(new_state_dict)
    return sam_dict

In [ ]:
%%writefile /content/SAM-Med2D/DataLoader.py
# (Viết trước training để TrainingDataset có _noisy_box + zoom_ratio)
import os
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
import numpy as np
from torch.nn import functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import train_transforms, get_boxes_from_mask, init_point_sampling
import json
import random

class TestingDataset(Dataset):
    def __init__(self, data_path, image_size=256, mode='test', requires_name=True, point_num=1, return_ori_mask=True, prompt_path=None, prompt_mode='zoom_out', zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        """
        Initializes a TestingDataset object.
        """
        self.image_size = image_size
        self.return_ori_mask = return_ori_mask
        self.prompt_path = prompt_path
        self.prompt_list = {} if prompt_path is None else json.load(open(prompt_path, "r"))
        self.requires_name = requires_name
        self.point_num = point_num
        self.mode = mode
        self.is_train = (mode == 'train')

        # Thêm biến điều khiển robust bbox
        self.prompt_mode = prompt_mode
        self.zoom_ratio = zoom_ratio
        self.shift_ratio = shift_ratio

        json_file = open(os.path.join(data_path, f'label2image_{mode}.json'), "r")
        dataset = json.load(json_file)

        sorted_items = sorted(dataset.items(), key=lambda x: os.path.basename(x[0]))
        self.label_paths = [k for k, v in sorted_items]
        self.image_paths = [v for k, v in sorted_items]

        self.pixel_mean = [123.675, 116.28, 103.53]
        self.pixel_std = [58.395, 57.12, 57.375]

    def _zoom_out_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        lo, hi = self.zoom_ratio
        if self.is_train:
            r_l, r_r = random.uniform(lo, hi), random.uniform(lo, hi)
            r_t, r_b = random.uniform(lo, hi), random.uniform(lo, hi)
        else:
            r = (lo + hi) / 2
            r_l = r_r = r_t = r_b = r
        bx_min = max(0,       x_min - gt_w * r_l)
        bx_max = min(orig_w,  x_max + gt_w * r_r)
        by_min = max(0,       y_min - gt_h * r_t)
        by_max = min(orig_h,  y_max + gt_h * r_b)
        return bx_min, bx_max, by_min, by_max

    def _shift_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=None):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)

        if self.is_train:
            dx = random.uniform(-gt_w * self.shift_ratio, gt_w * self.shift_ratio)
            dy = random.uniform(-gt_h * self.shift_ratio, gt_h * self.shift_ratio)
        else:
            rng = random.Random(seed_idx or 0)
            dx = rng.uniform(gt_w * 0.4, gt_w * 0.7) * self.shift_ratio
            dy = rng.uniform(gt_h * 0.1, gt_h * 0.3) * self.shift_ratio

        bx_min = max(0,       bx_min + dx)
        bx_max = min(orig_w,  bx_max + dx)
        by_min = max(0,       by_min + dy)
        by_max = min(orig_h,  by_max + dy)

        if min(bx_max, x_max) - max(bx_min, x_min) < gt_w * 0.3:
            if dx > 0: bx_max = min(orig_w, x_max + gt_w * 0.15)
            else:      bx_min = max(0,      x_min - gt_w * 0.15)
        if min(by_max, y_max) - max(by_min, y_min) < gt_h * 0.3:
            if dy > 0: by_max = min(orig_h, y_max + gt_h * 0.15)
            else:      by_min = max(0,      y_min - gt_h * 0.15)
        return bx_min, bx_max, by_min, by_max

    def __getitem__(self, index):
        image_input = {}
        try:
            image = cv2.imread(self.image_paths[index])
            image = (image - self.pixel_mean) / self.pixel_std
        except:
            print(self.image_paths[index])

        mask_path = self.label_paths[index]
        ori_np_mask = cv2.imread(mask_path, 0)

        if ori_np_mask.max() == 255:
            ori_np_mask = ori_np_mask / 255

        assert np.array_equal(ori_np_mask, ori_np_mask.astype(bool)), f"Mask should only contain binary values 0 and 1. {self.label_paths[index]}"

        h, w = ori_np_mask.shape
        ori_mask = torch.tensor(ori_np_mask).unsqueeze(0)

        transforms = train_transforms(self.image_size, h, w)
        augments = transforms(image=image, mask=ori_np_mask)
        image, mask = augments['image'], augments['mask'].to(torch.int64)

        if self.prompt_path is None:
            # Lấy bbox chặt từ mask đã transform
            y_indices, x_indices = torch.where(mask > 0)
            if len(y_indices) > 0 and len(x_indices) > 0:
                y_min, y_max = y_indices.min().item(), y_indices.max().item()
                x_min, x_max = x_indices.min().item(), x_indices.max().item()
            else:
                x_min, x_max, y_min, y_max = 0, self.image_size, 0, self.image_size

            orig_h, orig_w = self.image_size, self.image_size

            # Áp dụng các kịch bản nhiễu Bounding Box
            if self.prompt_mode == 'zoom_out':
                bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
            elif self.prompt_mode == 'shift':
                bx_min, bx_max, by_min, by_max = self._shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=index)
            elif self.prompt_mode == 'mixed_7_3':
                use_shift = (random.random() < 0.3) if self.is_train else (index % 10 >= 7)
                if use_shift:
                    bx_min, bx_max, by_min, by_max = self._shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=index)
                else:
                    bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
            else:
                bx_min, bx_max, by_min, by_max = x_min, x_max, y_min, y_max

            boxes = torch.tensor([[bx_min, by_min, bx_max, by_max]], dtype=torch.float)
            point_coords, point_labels = init_point_sampling(mask, self.point_num)
        else:
            prompt_key = mask_path.split('/')[-1]
            boxes = torch.as_tensor(self.prompt_list[prompt_key]["boxes"], dtype=torch.float)
            point_coords = torch.as_tensor(self.prompt_list[prompt_key]["point_coords"], dtype=torch.float)
            point_labels = torch.as_tensor(self.prompt_list[prompt_key]["point_labels"], dtype=torch.int)

        image_input["image"] = image
        image_input["label"] = mask.unsqueeze(0)
        image_input["point_coords"] = point_coords
        image_input["point_labels"] = point_labels
        image_input["boxes"] = boxes
        image_input["original_size"] = (h, w)
        image_input["label_path"] = '/'.join(mask_path.split('/')[:-1])

        if self.return_ori_mask:
            image_input["ori_label"] = ori_mask

        image_name = self.label_paths[index].split('/')[-1]
        if self.requires_name:
            image_input["name"] = image_name
            return image_input
        else:
            return image_input

    def __len__(self):
        return len(self.label_paths)

class TrainingDataset(Dataset):
    def __init__(self, data_dir, image_size=256, mode='train', requires_name=True, point_num=1, mask_num=5,
                 zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        self.image_size = image_size
        self.requires_name = requires_name
        self.point_num = point_num
        self.mask_num = mask_num
        self.zoom_ratio = zoom_ratio      # noisy bbox – cùng setup với PGA
        self.shift_ratio = shift_ratio
        self.pixel_mean = [123.675, 116.28, 103.53]
        self.pixel_std = [58.395, 57.12, 57.375]

        dataset = json.load(open(os.path.join(data_dir, f'image2label_{mode}.json'), "r"))
        self.image_paths = list(dataset.keys())
        self.label_paths = list(dataset.values())

    def _noisy_box(self, mask_tensor: torch.Tensor) -> torch.Tensor:
        """Tight bbox + zoom-out ngẫu nhiên [lo, hi] – giống PGA training."""
        y_idx, x_idx = torch.where(mask_tensor > 0)
        if len(y_idx) == 0:
            return torch.zeros(1, 4, dtype=torch.float)
        x0, x1 = int(x_idx.min()), int(x_idx.max())
        y0, y1 = int(y_idx.min()), int(y_idx.max())
        H = W = self.image_size
        lo, hi = self.zoom_ratio
        r_l = random.uniform(lo, hi); r_r = random.uniform(lo, hi)
        r_t = random.uniform(lo, hi); r_b = random.uniform(lo, hi)
        gw, gh = x1 - x0, y1 - y0
        bx0 = max(0, x0 - gw*r_l);  bx1 = min(W, x1 + gw*r_r)
        by0 = max(0, y0 - gh*r_t);  by1 = min(H, y1 + gh*r_b)
        return torch.tensor([[bx0, by0, bx1, by1]], dtype=torch.float)

    def __getitem__(self, index):
        image_input = {}
        try:
            image = cv2.imread(self.image_paths[index])
            image = (image - self.pixel_mean) / self.pixel_std
        except:
            print(self.image_paths[index])

        h, w, _ = image.shape
        transforms = train_transforms(self.image_size, h, w)

        masks_list = []
        boxes_list = []
        point_coords_list, point_labels_list = [], []
        mask_path = random.choices(self.label_paths[index], k=self.mask_num)
        for m in mask_path:
            pre_mask = cv2.imread(m, 0)
            if pre_mask.max() == 255:
                pre_mask = pre_mask / 255

            augments = transforms(image=image, mask=pre_mask)
            image_tensor, mask_tensor = augments['image'], augments['mask'].to(torch.int64)

            boxes = self._noisy_box(mask_tensor)
            point_coords, point_label = init_point_sampling(mask_tensor, self.point_num)

            masks_list.append(mask_tensor)
            boxes_list.append(boxes)
            point_coords_list.append(point_coords)
            point_labels_list.append(point_label)

        mask = torch.stack(masks_list, dim=0)
        boxes = torch.stack(boxes_list, dim=0)
        point_coords = torch.stack(point_coords_list, dim=0)
        point_labels = torch.stack(point_labels_list, dim=0)

        image_input["image"] = image_tensor.unsqueeze(0)
        image_input["label"] = mask.unsqueeze(1)
        image_input["boxes"] = boxes
        image_input["point_coords"] = point_coords
        image_input["point_labels"] = point_labels

        image_name = self.image_paths[index].split('/')[-1]
        if self.requires_name:
            image_input["name"] = image_name
            return image_input
        else:
            return image_input

    def __len__(self):
        return len(self.image_paths)

def stack_dict_batched(batched_input):
    out_dict = {}
    for k,v in batched_input.items():
        if isinstance(v, list):
            out_dict[k] = v
        else:
            out_dict[k] = v.reshape(-1, *v.shape[2:])
    return out_dict

In [ ]:
# Training — noisy bbox cùng setup PGA (công bằng)
# zoom_ratio=0.15-0.45 : giống PGA training bbox noise
# iter_point=3  : 3 lần refinement/batch
# mask_num=5    : 5 mask augmentation/ảnh
# batch_size=2  : tránh OOM với T4
# epochs=50, early_stop=10
%cd /content/SAM-Med2D
!python train.py \
    --work_dir "workdir" \
    --image_size 256 \
    --mask_num 5 \
    --data_path "dataset_BTXRD" \
    --sam_checkpoint "checkpoint/sam_med2d.pth" \
    --iter_point 3 \
    --encoder_adapter True \
    --epochs 50 \
    --early_stop 10 \
    --batch_size 2 \
    --zoom_ratio 0.15 0.45 \
    --shift_ratio 0.30

In [ ]:
import glob, os, shutil

# Lấy best và last checkpoint
best_ckpt = "/content/SAM-Med2D/workdir/models/sam-med2d/best_sam.pth"
last_ckpt = "/content/SAM-Med2D/workdir/models/sam-med2d/last_sam.pth"

os.makedirs("/content/drive/MyDrive/model", exist_ok=True)
for ckpt in [best_ckpt, last_ckpt]:
    if os.path.exists(ckpt):
        shutil.copy(ckpt, "/content/drive/MyDrive/model/")
        print(f"✅ Saved to Drive: {os.path.basename(ckpt)}")
    else:
        print(f"⚠️ Không tìm thấy: {os.path.basename(ckpt)}")

## Phần 2 – Thí nghiệm Đánh giá

Ba kịch bản prompt bbox được kiểm tra:
- **A – Zoom-Out ~30%**: bbox mở rộng đều 4 phía 30% so với tight bbox GT (tương ứng Gaussian heatmap của PGA)
- **B – Shift ~30%**: bbox lệch vị trí ~30% (mô phỏng người dùng chú thích không chính xác)
- **C – Mixed 70/30**: 70% zoom-out + 30% shift (kịch bản thực tế)

### 2.0 – Chuẩn bị (chạy 1 lần trước khi test)

In [ ]:
%cd /content/SAM-Med2D
import os, json, glob

# ── 1. Tạo label2image_test.json ────────────────────────────────────────────
def create_evaluation_json(split="test"):
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"
    label2image = {}

    if not os.path.exists(mask_dir) or not os.path.exists(img_dir):
        print(f"[!] Không tìm thấy thư mục cho tập: '{split}'"); return

    img_dict = {}
    for f in os.listdir(img_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_dict[os.path.splitext(f)[0]] = f

    for mask_name in sorted(os.listdir(mask_dir)):
        if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        mask_base = os.path.splitext(mask_name)[0]
        img_base  = mask_base.split("_")[0] if "_" in mask_base else mask_base
        if img_base in img_dict:
            label2image[f"dataset_BTXRD/{split}/masks/{mask_name}"] =                         f"dataset_BTXRD/{split}/images/{img_dict[img_base]}"

    out = f"dataset_BTXRD/label2image_{split}.json"
    with open(out, "w", encoding="utf-8") as f_out:
        json.dump(label2image, f_out, indent=4, ensure_ascii=False)
    print(f"[*] label2image_{split}.json — {len(label2image)} mẫu → {out}")

create_evaluation_json("test")
create_evaluation_json("val")

# ── 2. Load CHECKPOINT (ưu tiên best > last, Drive > local) ──────────────────
def find_ckpt(folder):
    for name in ["best_sam.pth", "last_sam.pth"]:
        p = os.path.join(folder, name)
        if os.path.exists(p): return p
    return None

CHECKPOINT = (find_ckpt("/content/drive/MyDrive/model") or
              find_ckpt("/content/SAM-Med2D/workdir/models/sam-med2d") or "")
if CHECKPOINT:
    src = "Drive" if "MyDrive" in CHECKPOINT else "Local"
    print(f"[{src}] ✅ {CHECKPOINT}")
else:
    print("❌ Không tìm thấy checkpoint. Chạy training trước.")

In [ ]:
%%writefile /content/SAM-Med2D/metrics.py
import torch
import torch.nn as nn


class SegMetrics(nn.Module):
    """
    SegMetrics(pred, label, metrics)
    pred, label : Tensor [B, 1, H, W]  (logits or 0/1)
    metrics     : list of str in {'iou','dice','precision','recall'}
    Usage       : bm = SegMetrics(pred, label, ['iou','dice','precision','recall'])
                  val = float(bm[i])
    """

    def __init__(self, pred, label, metrics):
        super().__init__()
        pred_bin  = (pred  > 0).float()
        label_bin = (label > 0).float()

        B = pred_bin.shape[0]
        p = pred_bin.view(B, -1)
        g = label_bin.view(B, -1)

        tp = (p * g).sum(dim=1)                 # [B]
        fp = (p * (1 - g)).sum(dim=1)
        fn = ((1 - p) * g).sum(dim=1)
        eps = 1e-7

        iou       = (tp / (tp + fp + fn + eps)).mean()
        dice      = (2 * tp / (2 * tp + fp + fn + eps)).mean()
        precision = (tp / (tp + fp + eps)).mean()
        recall    = (tp / (tp + fn + eps)).mean()

        self._map = {
            'iou':       iou,
            'dice':      dice,
            'precision': precision,
            'recall':    recall,
        }
        self._results = [self._map[m] for m in metrics]

    def __getitem__(self, idx):
        return self._results[idx]

    def __len__(self):
        return len(self._results)


In [ ]:
%%writefile /content/SAM-Med2D/test.py
from segment_anything import sam_model_registry
import torch.nn as nn
import torch
import argparse
import os
from utils import FocalDiceloss_IoULoss, generate_point, save_masks
from torch.utils.data import DataLoader
from DataLoader import TestingDataset
from metrics import SegMetrics
import time
from tqdm import tqdm
import numpy as np
from torch.nn import functional as F
import logging
import datetime
import cv2
import random
import csv
import json
from scipy.ndimage import binary_erosion, distance_transform_edt

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir", type=str, default="workdir", help="work dir")
    parser.add_argument("--run_name", type=str, default="sammed", help="run model name")
    parser.add_argument("--batch_size", type=int, default=1, help="batch size")
    parser.add_argument("--image_size", type=int, default=256, help="image_size")
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument("--data_path", type=str, default="data_demo", help="train data path")
    parser.add_argument("--metrics", nargs='+', default=['iou', 'dice', 'precision', 'recall'], help="metrics")
    parser.add_argument("--model_type", type=str, default="vit_b", help="sam model_type")
    parser.add_argument("--sam_checkpoint", type=str, default="pretrain_model/sam-med2d_b.pth", help="sam checkpoint")
    parser.add_argument("--boxes_prompt", type=bool, default=True, help="use boxes prompt")
    parser.add_argument("--point_num", type=int, default=1, help="point num")
    parser.add_argument("--iter_point", type=int, default=1, help="iter num")
    parser.add_argument("--multimask", type=bool, default=True, help="ouput multimask")
    parser.add_argument("--encoder_adapter", type=bool, default=True, help="use adapter")
    parser.add_argument("--prompt_path", type=str, default=None, help="fix prompt path")
    parser.add_argument("--save_pred", type=bool, default=False, help="save result")
    parser.add_argument("--prompt_mode", type=str, default="zoom_out",
                        choices=['zoom_out', 'shift', 'mixed_7_3'],
                        help="Bbox prompt scenario")
    parser.add_argument("--zoom_ratio", type=float, nargs=2, default=[0.15, 0.45])
    parser.add_argument("--shift_ratio", type=float, default=0.30)
    args = parser.parse_args()
    if args.iter_point > 1:
        args.point_num = 1
    return args

def to_device(batch_input, device):
    device_input = {}
    for key, value in batch_input.items():
        if value is not None:
            if key=='image' or key=='label':
                device_input[key] = value.float().to(device)
            elif type(value) is list or type(value) is torch.Size:
                device_input[key] = value
            else:
                device_input[key] = value.to(device)
        else:
            device_input[key] = value
    return device_input

def postprocess_masks(low_res_masks, image_size, original_size):
    ori_h, ori_w = original_size
    masks = F.interpolate(low_res_masks, (image_size, image_size),
                          mode="bilinear", align_corners=False)
    if ori_h < image_size and ori_w < image_size:
        top  = torch.div((image_size - ori_h), 2, rounding_mode='trunc')
        left = torch.div((image_size - ori_w), 2, rounding_mode='trunc')
        masks = masks[..., top:ori_h+top, left:ori_w+left]
        pad = (top, left)
    else:
        masks = F.interpolate(masks, original_size, mode="bilinear", align_corners=False)
        pad = None
    return masks, pad

def prompt_and_decoder(args, batched_input, ddp_model, image_embeddings):
    points = (batched_input["point_coords"], batched_input["point_labels"]) \
             if batched_input["point_coords"] is not None else None
    with torch.no_grad():
        sparse_embeddings, dense_embeddings = ddp_model.prompt_encoder(
            points=points,
            boxes=batched_input.get("boxes", None),
            masks=batched_input.get("mask_inputs", None),
        )
        low_res_masks, iou_predictions = ddp_model.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=ddp_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=args.multimask,
        )
    if args.multimask:
        max_values, max_indexs = torch.max(iou_predictions, dim=1)
        iou_predictions = max_values.unsqueeze(1)
        low_res_masks = torch.stack([low_res_masks[i:i+1, idx]
                                     for i, idx in enumerate(max_indexs)], 0)
    masks = F.interpolate(low_res_masks, (args.image_size, args.image_size),
                          mode="bilinear", align_corners=False)
    return masks, low_res_masks, iou_predictions

def extract_lcc(binary_map):
    if binary_map.sum() == 0: return binary_map
    mask_u8 = binary_map.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if n <= 1: return binary_map
    return (labels == (1 + np.argmax(stats[1:, cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return 256.0
    pe, ge = pred ^ binary_erosion(pred), gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return 256.0
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred_bin, gt_bin):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d / gt_diag, 0.0, 1.0))

def main(args):
    print('*'*80)
    for k, v in vars(args).items(): print(f"  {k}: {v}")
    print('*'*80)

    model = sam_model_registry[args.model_type](args).to(args.device)
    criterion = FocalDiceloss_IoULoss()
    test_dataset = TestingDataset(
        data_path=args.data_path, image_size=args.image_size,
        mode='test', requires_name=True, point_num=args.point_num,
        return_ori_mask=True, prompt_path=args.prompt_path,
        prompt_mode=args.prompt_mode,
        zoom_ratio=tuple(args.zoom_ratio), shift_ratio=args.shift_ratio)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)
    print(f'Test data: {len(test_loader)}  |  prompt_mode: {args.prompt_mode}')

    model.eval()
    test_loss, test_hd95, test_cbl = [], [], []
    test_iter_metrics = [0] * len(args.metrics)
    l = len(test_loader)

    for batched_input in tqdm(test_loader):
        batched_input = to_device(batched_input, args.device)
        ori_labels   = batched_input["ori_label"]
        original_size = batched_input["original_size"]
        img_name     = batched_input['name'][0]

        with torch.no_grad():
            image_embeddings = model.image_encoder(batched_input["image"])

        batched_input["point_coords"], batched_input["point_labels"] = None, None
        save_path = os.path.join(args.work_dir, args.run_name, f"boxes_{args.prompt_mode}")
        masks, low_res_masks, iou_predictions = prompt_and_decoder(
            args, batched_input, model, image_embeddings)

        masks, pad = postprocess_masks(low_res_masks, args.image_size, original_size)
        if args.save_pred:
            save_masks(masks, save_path, img_name, args.image_size,
                       original_size, pad, batched_input.get("boxes", None), None)

        loss = criterion(masks, ori_labels, iou_predictions)
        test_loss.append(loss.item())

        pred_bin = extract_lcc((masks[0, 0].cpu().numpy() > 0.0).astype(np.float32))
        gt_bin   = ori_labels[0, 0].cpu().numpy().astype(np.float32)
        test_hd95.append(calc_hd95(pred_bin, gt_bin))
        cbl = calc_cbl(pred_bin, gt_bin)
        if cbl is not None: test_cbl.append(cbl)

        bm = SegMetrics(masks, ori_labels, args.metrics)
        for j in range(len(args.metrics)):
            test_iter_metrics[j] += float(bm[j])

    metrics = {args.metrics[i]: f"{test_iter_metrics[i]/l:.4f}" for i in range(len(args.metrics))}
    mean_dice = test_iter_metrics[args.metrics.index('dice')] / l if 'dice' in args.metrics else 0.0
    mean_iou  = test_iter_metrics[args.metrics.index('iou')]  / l if 'iou'  in args.metrics else 0.0
    mean_pre  = test_iter_metrics[args.metrics.index('precision')] / l if 'precision' in args.metrics else 0.0
    mean_rec  = test_iter_metrics[args.metrics.index('recall')]    / l if 'recall'    in args.metrics else 0.0
    mean_hd95 = float(np.mean(test_hd95)) if test_hd95 else 0.0
    mean_cbl  = float(np.mean(test_cbl))  if test_cbl  else 0.0
    print(f"\n[{args.prompt_mode}] loss={np.mean(test_loss):.4f} | "
          f"IoU={mean_iou:.4f} | Dice={mean_dice:.4f} | "
          f"Pre={mean_pre:.4f} | Rec={mean_rec:.4f} | "
          f"HD95={mean_hd95:.2f}px | CBL={mean_cbl:.4f} | N={l}")

    # Luu CSV
    os.makedirs(os.path.join(args.work_dir, "csv"), exist_ok=True)
    csv_path = os.path.join(args.work_dir, "csv", f"sammed2d_{args.prompt_mode}.csv")
    with open(csv_path, "w", newline="") as f_csv:
        writer = csv.writer(f_csv)
        writer.writerow(["model", "prompt_mode", "dice", "iou", "precision", "recall", "hd95", "cbl", "n_samples"])
        writer.writerow(["SAM-Med2D", args.prompt_mode,
                         f"{mean_dice:.4f}", f"{mean_iou:.4f}",
                         f"{mean_pre:.4f}", f"{mean_rec:.4f}",
                         f"{mean_hd95:.4f}", f"{mean_cbl:.4f}", l])
    print(f"CSV saved: {csv_path}")

if __name__ == '__main__':
    args = parse_args()
    main(args)

### 2.1 – Kịch bản A: Zoom-Out (~30%)
Bbox được mở rộng đều 4 phía 30% so với tight bbox GT — tương đương Gaussian heatmap "đúng" của PGA.

In [ ]:
assert CHECKPOINT, "❌ CHECKPOINT rỗng — chạy cell 12 trước"
!python test.py \
    --work_dir "workdir/test_results" \
    --data_path "dataset_BTXRD" \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode zoom_out \
    --sam_checkpoint {CHECKPOINT} \
    --save_pred True

### 2.2 – Kịch bản B: Shift (~30%)
Bbox bị lệch vị trí ~30% so với GT — mô phỏng người dùng chú thích không chính xác.

In [ ]:
assert CHECKPOINT, "❌ CHECKPOINT rỗng — chạy cell 12 trước"
!python test.py \
    --work_dir "workdir/test_results" \
    --data_path "dataset_BTXRD" \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode shift \
    --sam_checkpoint {CHECKPOINT} \
    --save_pred True

### 2.3 – Kịch bản C: Mixed 70/30
70% zoom-out + 30% shift — kịch bản thực tế người dùng.

In [ ]:
assert CHECKPOINT, "❌ CHECKPOINT rỗng — chạy cell 12 trước"
!python test.py \
    --work_dir "workdir/test_results" \
    --data_path "dataset_BTXRD" \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode mixed_7_3 \
    --sam_checkpoint {CHECKPOINT} \
    --save_pred True

### 2.4 – Tổng hợp kết quả & Visualization
Bảng so sánh 3 kịch bản + hiển thị trực quan trên 3 ảnh đại diện (cùng ảnh với PGA).

In [ ]:
# ── Tổng hợp & Visualization 3 kịch bản (PGA-style) ─────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import cv2, numpy as np, json, os, csv, random

# ── Cùng 3 ảnh như PGA ────────────────────────────────────────────────────────
# Tự động lấy 10 ảnh đầu theo thứ tự từ test set
# VIS_IMAGES được xây dựng sau khi load label2image (xem bên dưới)

MODES       = ['zoom_out',   'shift',     'mixed_7_3']
MODE_LABELS = ['Zoom-Out\n(~30%)', 'Shift\n(~30%)', 'Mixed\n70/30']
MODE_COLORS = ['limegreen',  'tomato',    'gold']
PRED_DIRS   = {m: f"workdir/test_results/sammed/boxes_{m}" for m in MODES}
CSV_DIR     = "workdir/test_results/csv"

# ════════════════════════════════════════════════════════════════════════════
# 1. BẢNG SỐ LIỆU
# ════════════════════════════════════════════════════════════════════════════
metrics_table = {}
METRIC_KEYS = ['iou', 'dice', 'precision', 'recall', 'hd95', 'cbl']

for mode in MODES:
    csv_path = os.path.join(CSV_DIR, f"sammed2d_{mode}.csv")
    if os.path.exists(csv_path):
        rows = list(csv.DictReader(open(csv_path)))
        if rows:
            last = rows[-1]  # dòng summary cuối
            metrics_table[mode] = {k: last.get(k, 'N/A') for k in METRIC_KEYS}
        else:
            metrics_table[mode] = {k: 'N/A' for k in METRIC_KEYS}
    else:
        metrics_table[mode] = {k: 'N/A' for k in METRIC_KEYS}

# In bảng
header = f"{'Kịch bản':<16} {'Dice':>8} {'IoU':>8} {'Prec':>8} {'Recall':>8} {'HD95':>8} {'CBL':>8}"
print("\n" + "="*78)
print("  SAM-Med2D — Kết quả 3 kịch bản prompt bbox")
print("="*70)
print(header)
print("-"*78)
MODE_DISPLAY = {'zoom_out': 'A. Zoom-Out', 'shift': 'B. Shift', 'mixed_7_3': 'C. Mixed 70/30'}
for mode in MODES:
    m = metrics_table[mode]
    def fmt(v):
        try: return f"{float(v):.4f}"
        except: return str(v)
    print(f"  {MODE_DISPLAY[mode]:<14} {fmt(m['dice']):>8} {fmt(m['iou']):>8} "
          f"{fmt(m['precision']):>8} {fmt(m['recall']):>8} {fmt(m['hd95']):>8} {fmt(m['cbl']):>8}")
print("="*78 + "\n")

# ════════════════════════════════════════════════════════════════════════════
# 2. HELPERS BBOX (y hệt DataLoader – test-time deterministic)
# ════════════════════════════════════════════════════════════════════════════
def _zoom_out(x0, x1, y0, y1, H, W, r=0.30):
    gw, gh = x1-x0, y1-y0
    return (max(0,x0-gw*r), min(W,x1+gw*r), max(0,y0-gh*r), min(H,y1+gh*r))

def _shift(x0, x1, y0, y1, H, W, shift_ratio=0.30, seed=0):
    bx0,bx1,by0,by1 = _zoom_out(x0,x1,y0,y1,H,W)
    gw, gh = x1-x0, y1-y0
    rng = random.Random(seed)
    dx = rng.uniform(gw*0.4, gw*0.7) * shift_ratio
    dy = rng.uniform(gh*0.1, gh*0.3) * shift_ratio
    bx0=max(0,bx0+dx); bx1=min(W,bx1+dx)
    by0=max(0,by0+dy); by1=min(H,by1+dy)
    if min(bx1,x1)-max(bx0,x0) < gw*0.3: bx1=min(W,x1+gw*0.15)
    if min(by1,y1)-max(by0,y0) < gh*0.3: by1=min(H,y1+gh*0.15)
    return bx0,bx1,by0,by1

def get_bbox_for_mode(gt_mask, mode, idx=0):
    ys,xs = np.where(gt_mask>0)
    if len(xs)==0: return None
    x0,x1,y0,y1 = int(xs.min()),int(xs.max()),int(ys.min()),int(ys.max())
    H,W = gt_mask.shape
    if mode=='zoom_out':
        bx0,bx1,by0,by1 = _zoom_out(x0,x1,y0,y1,H,W)
    elif mode=='shift':
        bx0,bx1,by0,by1 = _shift(x0,x1,y0,y1,H,W,seed=idx)
    else:  # mixed_7_3: idx%10>=7 → shift
        if idx%10>=7: bx0,bx1,by0,by1 = _shift(x0,x1,y0,y1,H,W,seed=idx)
        else:         bx0,bx1,by0,by1 = _zoom_out(x0,x1,y0,y1,H,W)
    return float(bx0),float(by0),float(bx1-bx0),float(by1-by0)  # x,y,w,h

# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD label2image → tìm mask cho mỗi VIS_IMAGE
# ════════════════════════════════════════════════════════════════════════════
with open("dataset_BTXRD/label2image_test.json") as f:
    label2image = json.load(f)

# 10 sample đầu theo thứ tự polygon (per-sample, không dedup)
VIS_IMAGES = sorted(os.path.basename(m) for m in label2image.keys())[:10]

vis_samples = {}  # mask_basename → (img_path, mask_path)
for mask_path, img_path in label2image.items():
    mask_name = os.path.basename(mask_path)
    if mask_name in VIS_IMAGES:
        vis_samples[mask_name] = (img_path, mask_path)

missing = [v for v in VIS_IMAGES if v not in vis_samples]
if missing:
    print(f"⚠️  Không tìm thấy trong label2image: {missing}")

# ════════════════════════════════════════════════════════════════════════════
# 4. VISUALIZATION – 1 figure per ảnh, 3 hàng × 4 cột
# ════════════════════════════════════════════════════════════════════════════
for vis_name in VIS_IMAGES:
    if vis_name not in vis_samples:
        print(f"⏭  Bỏ qua {vis_name} (không có trong test set)")
        continue

    img_path, mask_path = vis_samples[vis_name]
    img  = cv2.resize(cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE), (256,256))
    gt   = cv2.resize(cv2.imread(mask_path, 0), (256,256), interpolation=cv2.INTER_NEAREST)
    if gt.max()==255: gt=(gt/255).astype(np.uint8)

    rgb   = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    gt_ov = rgb.copy()
    gt_ov[gt>0] = np.clip(rgb[gt>0]*0.4 + np.array([0,200,0])*0.6, 0, 255)

    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    fig.suptitle(f"SAM-Med2D — {vis_name} — So sánh 3 kịch bản prompt",
                 fontsize=13, fontweight='bold', y=1.01)

    for row, (mode, label, color) in enumerate(zip(MODES, MODE_LABELS, MODE_COLORS)):
        pred_file = os.path.join(PRED_DIRS[mode], os.path.basename(mask_path))

        # Load prediction (hoặc blank nếu chưa có)
        if os.path.exists(pred_file):
            pred = cv2.resize(cv2.imread(pred_file, 0), (256,256),
                              interpolation=cv2.INTER_NEAREST)
            if pred.max()==255: pred=(pred/255).astype(np.uint8)
            pred = (pred>0).astype(np.uint8)
        else:
            pred = np.zeros((256,256), dtype=np.uint8)

        pr_ov = rgb.copy()
        pr_ov[pred>0] = np.clip(rgb[pred>0]*0.4 + np.array([220,60,60])*0.6, 0, 255)

        # Diff: xanh=GT-only, đỏ=Pred-only, vàng=overlap
        diff = np.zeros((256,256,3), dtype=np.uint8)
        diff[gt>0]               = [0, 200, 0]
        diff[pred>0]             = [200, 60, 60]
        diff[(gt>0)&(pred>0)]    = [220, 200, 0]

        # Col 0 – ảnh gốc + bbox prompt (màu theo mode)
        axes[row,0].imshow(img, cmap='gray')
        bbox = get_bbox_for_mode(gt, mode, idx=row)
        if bbox:
            rect = Rectangle((bbox[0],bbox[1]),bbox[2],bbox[3],
                              linewidth=2.5, edgecolor=color, facecolor='none')
            axes[row,0].add_patch(rect)
        axes[row,0].set_ylabel(label, fontsize=11, fontweight='bold',
                               color=color, rotation=0, labelpad=65, va='center')
        if row==0: axes[row,0].set_title('Bbox Prompt', fontsize=9, fontweight='bold')

        # Col 1 – GT mask
        axes[row,1].imshow(gt_ov)
        if row==0: axes[row,1].set_title('Ground Truth (xanh)', fontsize=9, fontweight='bold')

        # Col 2 – SAMmed2D prediction
        axes[row,2].imshow(pr_ov)
        if row==0: axes[row,2].set_title('SAM-Med2D (đỏ)', fontsize=9, fontweight='bold')

        # Col 3 – Overlap diff
        axes[row,3].imshow(diff)
        if row==0: axes[row,3].set_title('GT / Pred / Overlap', fontsize=9, fontweight='bold')

        for ax in axes[row]: ax.axis('off')

    plt.tight_layout()
    out = f"visualization_sammed_{os.path.splitext(vis_name)[0]}.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✅ {out}")
